In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

PROJECT_ROOT = next(
    p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (p / 'date').is_dir() and (p / 'images').is_dir()
)
DATA_DIR = PROJECT_ROOT / 'date'
IMAGES_DIR = PROJECT_ROOT / 'images'

sns.set_theme(style="whitegrid",font_scale = 1.2)

plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

In [2]:
df = pd.read_parquet(DATA_DIR / 'user_behavior_cleaned_dataset.parquet')
df.head()

,user_id,item_id,category_id,behavior_type,timestamp,date
494,114726,2014874,1901483,pv,2017-11-24 16:00:00,2017-11-25
495,108891,4295341,3855599,pv,2017-11-24 16:00:01,2017-11-25
496,127538,1849003,2072473,pv,2017-11-24 16:00:01,2017-11-25
497,104088,4539468,3702593,pv,2017-11-24 16:00:02,2017-11-25
498,111942,1005979,2735466,pv,2017-11-24 16:00:02,2017-11-25


分析被购买最多次数的商品类目

In [3]:
# 按照品类分组后找到最后欢迎的商品类目前十
df_buy = df[df['behavior_type'] == 'buy'].groupby('category_id').size().reset_index(name='buy_count')
buy_top10 = df_buy.sort_values(by='buy_count',ascending = False).head(10)
print(f"购买量前十{buy_top10}")

购买量前十      category_id  buy_count
1375      2735466        385
713       1464116        363
2087      4145813        339
1444      2885642        326
2399      4801426        299
2379      4756105        272
505        982926        246
1495      3002561        184
1321      2640118        180
635       1320293        178


分析被浏览次数最多的商品

In [4]:
#按照浏览量分组查询浏览次数前十
df_pv = df[df['behavior_type'] == 'pv'].groupby('item_id').size().reset_index(name='pv_count')
pv_top10 = df_pv.sort_values(by='pv_count',ascending = False).head(10)
print(f"浏览量前十{pv_top10}")

浏览量前十        item_id  pv_count
61780    812879       298
10759    138964       233
293408  3845720       231
282759  3708121       205
154897  2032668       198
177804  2331370       198
178297  2338453       187
231320  3031354       179
117076  1535294       177
257197  3371523       170


复购率

In [5]:
#先计算每个用户的下单总量
user_buy_count = df[df['behavior_type'] == 'buy'].groupby(['user_id']).size().reset_index(name='buy_count')
#再计算购买次数大于等于2的用户
repurchase_user = user_buy_count[user_buy_count['buy_count'] >= 2]
# 计算复购率
repeat_purchase = len(repurchase_user)*100/len(user_buy_count)

print(user_buy_count)
print(repurchase_user)
print(f"复购率为{repeat_purchase:.2f}%")

      user_id  buy_count
0         100          8
1         117         10
2         119          3
3         121          1
4         122          3
...       ...        ...
6995  1017960          3
6996  1017965          1
6997  1017972          4
6998  1017997          2
6999  1018011          1

[7000 rows x 2 columns]
      user_id  buy_count
0         100          8
1         117         10
2         119          3
4         122          3
7        1072          3
...       ...        ...
6993  1017938          5
6994  1017958          2
6995  1017960          3
6997  1017972          4
6998  1017997          2

[4639 rows x 2 columns]
复购率为66.27%


复购率显著高于行业正常值,可能是数据集中仅包含限定日期内的活跃用户,并不包含流失用户,只能反映高活跃用户的购买粘性,不具有大盘代表性

重新计算可能合理的复购率,只计算核心日期前(11月25日)才开始购买第一次的用户,并且计算限定窗口为七天计算复购率

In [6]:
df1 = df[df['behavior_type'] == 'buy'].copy()
start_date = '2017-11-25'
df1 = df1[df1['date'] >= start_date]
first_buy_date = df1.groupby('user_id')['date'].min().rename('first_date')
df1 = df1.merge(first_buy_date, on='user_id', how='left')
df1['days_gap'] = (df1['date'] - df1['first_date']).dt.days

total_new_user = df1['user_id'].nunique()
new_repurchase_user = df1[df1['days_gap'].between(1, 7)]['user_id'].nunique()
new_repeat_purchase = new_repurchase_user*100/total_new_user

print(f"新用户总数: {total_new_user}")
print(f"7天内复购人数: {new_repurchase_user}")
print(f"保守复购率: {new_repeat_purchase:.2f}%")

新用户总数: 7000
7天内复购人数: 3806
保守复购率: 54.37%
